# LightGBM Model Development and Evaluation Script

---

## Modeling Pipeline

- **Baseline Model Development:** A standard model was trained using default/reasonable hyperparameters to establish a performance benchmark.

- **Hyperparameter Optimization:** Each baseline model was optimized using two complementary techniques:
  - ***`Optuna`***: Efficient, adaptive search with pruning for rapid convergence to high-performing configurations.
  - ***`RandomizedSearchCV`***: Sklearn-compatible baseline optimization for fair comparison and reproducibility.

- **Cross-Validation and Robustness Assessment:** Each model variant was evaluated using ***`TimeSeriesSplit`*** to preserve temporal order and prevent look-ahead bias. Metrics were aggregated across folds to assess stability.

- **Overfitting Analysis:** A detailed comparison between cross-validation metrics and test set results was conducted. Additional metrics, including ***`RMSE ratio`*** and ***`R² gap`***, were computed to quantify overfitting and assess model generalization. ***`Directional accuracy`*** and ***`financial metrics`*** (Sharpe Ratio, Max Drawdown) were also calculated for trading-relevant evaluation.

---

## Persisted Artifacts

To ensure reproducibility, transparency, and extendability, the following artifacts have been saved for **each model**:

- **Optimized Model Performance:** Individual CSV files capturing the performance of each model variant:
    - ***LightGBM (Baseline)***
    - ***LightGBM (Optuna)***
    - ***LightGBM (RandomizedSearchCV)***

- **Best Variation Performance:** A CSV file containing only the metrics of the best-performing variation per model.

- **Summary of Model Performance:** A consolidated, extendable CSV file (`AllModel_OverallPerformance.csv`) including:
    - Cross-validation results (`CV MSE`, `CV MAE`, `CV RMSE`, `CV R²`, `CV MAPE`)
    - Test set results (`Test MSE`, `Test MAE`, `Test RMSE`, `Test R²`, `Test MAPE`)
    - Financial metrics (`Sharpe Ratio`, `Sortino Ratio`, `Max Drawdown`, `Directional Accuracy`)
    - Overfitting metrics (`R² gap`, `RMSE ratio`)
    - Overfitting status and model generalization label

- **Overfitting DataFrame:** An extendable CSV (`AllModel_OverfittingAnalysis.csv`) capturing overfitting analysis metrics across all models and variations.

- **Best Model per Algorithm:** The serialized best-performing variant of each algorithm for ensemble consideration or deployment.

- **Model Comparison:** A summary notebook or script that loads `AllModel_OverallPerformance.csv` and generates publication-ready comparison visualizations.

Together, these artifacts provide a complete, reproducible record of the modeling process, facilitating model tracking, comparison, selection, and deployment.

## Import Libraries and Root Configuration

In [1]:
""" Configure the utilities module path for imports """
import sys
import os
from pathlib import Path

# get project root as parent of current working directory
PROJECT_ROOT = Path(os.getcwd()).parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
# artifacts root
DATA_ROOT = PROJECT_ROOT / "data"
FEATURE_ROOT = PROJECT_ROOT / "artifacts" / "FeatureSelection"
FIGURE_ROOT = PROJECT_ROOT / "visualizations" / "ModelEvaluation"
MODEL_ROOT = PROJECT_ROOT / "artifacts" / "Models"
PERFORMANCE_ROOT = PROJECT_ROOT / "artifacts" / "ModelPerformance"

In [3]:
# records and calculations
import pandas as pd
import numpy as np

# avoid minor warnings
import warnings
warnings.filterwarnings('ignore')

# visualizations
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error
from scipy.stats import uniform, randint, loguniform

# gradient boosting model
import lightgbm as lgb

# optimization
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# import helper functions
from src.utilities import Evaluator, DataHandler, ModelPersister

## Load Dataset and Artifacts

In [4]:
df, x, y = DataHandler.load_dataset(DATA_ROOT / "gold_price_engineered.csv", target_col="target")
artifacts = DataHandler.load_artifacts(FEATURE_ROOT, cv_method='tscv')

In [5]:
# check dataset loading
df.head()

,Date,SPX_Return,GLD_Return,USO_Return,SLV_Return,EURUSD_Return,SPX_Return_lag1,SPX_Return_lag2,SPX_Return_lag3,SPX_Return_lag4,...,EURUSD_Return_lag2,EURUSD_Return_lag3,EURUSD_Return_lag4,EURUSD_Return_lag5,GLD_Return_lag1,GLD_Return_lag2,GLD_Return_lag3,GLD_Return_lag4,GLD_Return_lag5,target
0,2008-01-10,0.007948,0.019642,-0.016346,0.034858,0.009339,-0.002650,0.023711,-0.004229,-0.005142,...,0.023711,-0.004229,-0.005142,0.008367,-0.002650,0.023711,-0.004229,-0.005142,0.008367,0.003739
1,2008-01-11,-0.013595,0.003739,-0.012564,0.000996,-0.000739,0.019642,-0.002650,0.023711,-0.004229,...,-0.002650,0.023711,-0.004229,-0.005142,0.019642,-0.002650,0.023711,-0.004229,-0.005142,0.010838
2,2008-01-14,0.010871,0.010838,0.015871,0.012627,0.005337,0.003739,0.019642,-0.002650,0.023711,...,0.019642,-0.002650,0.023711,-0.004229,0.003739,0.019642,-0.002650,0.023711,-0.004229,-0.017311
3,2008-01-15,-0.024925,-0.017311,-0.019798,-0.027396,-0.004499,0.010838,0.003739,0.019642,-0.002650,...,0.003739,0.019642,-0.002650,0.023711,0.010838,0.003739,0.019642,-0.002650,0.023711,-0.014661
4,2008-01-16,-0.005612,-0.014661,-0.012778,-0.011368,-0.009326,-0.017311,0.010838,0.003739,0.019642,...,0.010838,0.003739,0.019642,-0.002650,-0.017311,0.010838,0.003739,0.019642,-0.002650,-0.002307


In [6]:
# check input features
x.head()

,Date,SPX_Return,GLD_Return,USO_Return,SLV_Return,EURUSD_Return,SPX_Return_lag1,SPX_Return_lag2,SPX_Return_lag3,SPX_Return_lag4,...,EURUSD_Return_lag1,EURUSD_Return_lag2,EURUSD_Return_lag3,EURUSD_Return_lag4,EURUSD_Return_lag5,GLD_Return_lag1,GLD_Return_lag2,GLD_Return_lag3,GLD_Return_lag4,GLD_Return_lag5
0,2008-01-10,0.007948,0.019642,-0.016346,0.034858,0.009339,-0.002650,0.023711,-0.004229,-0.005142,...,-0.002650,0.023711,-0.004229,-0.005142,0.008367,-0.002650,0.023711,-0.004229,-0.005142,0.008367
1,2008-01-11,-0.013595,0.003739,-0.012564,0.000996,-0.000739,0.019642,-0.002650,0.023711,-0.004229,...,0.019642,-0.002650,0.023711,-0.004229,-0.005142,0.019642,-0.002650,0.023711,-0.004229,-0.005142
2,2008-01-14,0.010871,0.010838,0.015871,0.012627,0.005337,0.003739,0.019642,-0.002650,0.023711,...,0.003739,0.019642,-0.002650,0.023711,-0.004229,0.003739,0.019642,-0.002650,0.023711,-0.004229
3,2008-01-15,-0.024925,-0.017311,-0.019798,-0.027396,-0.004499,0.010838,0.003739,0.019642,-0.002650,...,0.010838,0.003739,0.019642,-0.002650,0.023711,0.010838,0.003739,0.019642,-0.002650,0.023711
4,2008-01-16,-0.005612,-0.014661,-0.012778,-0.011368,-0.009326,-0.017311,0.010838,0.003739,0.019642,...,-0.017311,0.010838,0.003739,0.019642,-0.002650,-0.017311,0.010838,0.003739,0.019642,-0.002650


In [7]:
# check target feature
y.head()

0    0.003739
1    0.010838
2   -0.017311
3   -0.014661
4   -0.002307
Name: target, dtype: float64

In [8]:
# load train/test split data
x_train, x_test, y_train, y_test = artifacts['x_train'], artifacts['x_test'], artifacts['y_train'], artifacts['y_test']
cv = artifacts['cv']